# Foco-LLM: especialista em dedução (Qwen2.5-7B)
Treino pós-hoc em 80 casos de dedução. Comparação pareada do modelo base e do adaptador em 120 casos novos e disjuntos: dedução e rastreamento. Os arquivos completos ficam na saída da versão salva.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ["MPLBACKEND"] = "Agg"
if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError(f"Python 3.12 or 3.13 required; found {sys.version}")
if shutil.which("poetry") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "poetry==2.4.1"], check=True)
SOURCE_REF = "feat/deduction-specialist-7b"
repository = Path("/kaggle/temp/foco-llm-deduction")
output = Path("/kaggle/working/deduction-7b")
subprocess.run(
    [
        "git",
        "clone",
        "--branch",
        SOURCE_REF,
        "--single-branch",
        "https://github.com/Mathwesm/foco-llm.git",
        str(repository),
    ],
    check=True,
)
PINNED_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=repository, text=True
).strip()
subprocess.run(
    ["poetry", "install", "--only", "main,inference", "--no-interaction"],
    cwd=repository,
    check=True,
)
print("Pinned source:", PINNED_COMMIT, flush=True)

In [ ]:
command = ["poetry", "run", "python", "scripts/run_deduction_specialist.py"]
subprocess.run([*command, "prepare", "--output", str(output)], cwd=repository, check=True)
manifest = json.loads((output / "dataset-manifest.json").read_text(encoding="utf-8"))
for name, key in [
    ("training-dataset.json", "training_sha256"),
    ("holdout-dataset.json", "holdout_sha256"),
]:
    observed = hashlib.sha256((output / name).read_bytes()).hexdigest()
    if observed != manifest[key]:
        raise ValueError(f"Dataset integrity mismatch: {name}")
print(
    "Frozen train/holdout:",
    len(manifest["training_example_ids"]),
    manifest["holdout_cases"],
    flush=True,
)
subprocess.run(
    [
        "poetry",
        "run",
        "python",
        "scripts/run_cross_task_transfer.py",
        "evaluate",
        "--dataset",
        str(output / "holdout-dataset.json"),
        "--output",
        str(output / "baseline"),
        "--backend",
        "cloud",
    ],
    cwd=repository,
    check=True,
)
print("Base 7B evaluation complete", flush=True)

In [ ]:
subprocess.run(
    [*command, "train", "--output", str(output), "--checkpoint-every", "80"],
    cwd=repository,
    check=True,
)
completed = sorted((output / "training").glob("*/summary.json"))
if len(completed) != 1:
    raise RuntimeError(f"Expected one completed specialist run; found {len(completed)}")
adapter = completed[0].parent / "step-0320"
if not (adapter / "complete.json").is_file():
    raise RuntimeError("Step-0320 checkpoint is incomplete")
subprocess.run(
    [
        "poetry",
        "run",
        "python",
        "scripts/run_cross_task_transfer.py",
        "evaluate",
        "--dataset",
        str(output / "holdout-dataset.json"),
        "--output",
        str(output / "specialist"),
        "--backend",
        "cloud",
        "--adapter",
        str(adapter),
        "--training-dataset",
        str(output / "training-dataset.json"),
    ],
    cwd=repository,
    check=True,
)


def read_one(root):
    """Read the sole evaluation summary for one model arm."""
    paths = sorted(root.glob("*/summary.json"))
    if len(paths) != 1:
        raise RuntimeError(f"Expected one evaluation summary in {root}; found {len(paths)}")
    return json.loads(paths[0].read_text(encoding="utf-8"))


comparison = {
    "protocol": "deduction-specialist-7b-v1",
    "source_commit": PINNED_COMMIT,
    "dataset_manifest": manifest,
    "training": json.loads(completed[0].read_text(encoding="utf-8")),
    "base": read_one(output / "baseline"),
    "deduction_specialist": read_one(output / "specialist"),
}
(output / "comparison.json").write_text(json.dumps(comparison, indent=2), encoding="utf-8")
print(json.dumps(comparison, indent=2), flush=True)